In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# المحاكاة الدقيقة باستخدام primitives في Qiskit SDK

<details>
<summary><b>إصدارات الحزم</b></summary>

الكود في هذه الصفحة طُوِّر باستخدام المتطلبات التالية.
ننصح باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.4.0
```
</details>

تُنفِّذ الـ primitives المرجعية في Qiskit SDK محاكاة statevector محلية. لا تدعم هذه المحاكاة نمذجة ضوضاء الأجهزة، لكنها مفيدة لنمذجة الخوارزميات بسرعة قبل الانتقال إلى تقنيات محاكاة أكثر تقدمًا ([باستخدام Qiskit Aer](./simulate-stabilizer-circuits)) أو التشغيل على أجهزة حقيقية ([Qiskit Runtime primitives](primitives)).

يستطيع الـ Estimator primitive حساب قيم التوقع للدوائر، ويستطيع الـ Sampler primitive أخذ عينات من توزيعات المخرجات للدوائر.

تشرح الأقسام التالية كيفية استخدام الـ primitives المرجعية لتشغيل سير عملك محليًا.

## استخدام Estimator المرجعي
التطبيق المرجعي لـ `EstimatorV2` في `qiskit.primitives` الذي يعمل على محاكيات statevector محلية هو فئة [`StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator). يمكنه قبول الدوائر والـ observables والمعاملات كمدخلات، ويُعيد قيم التوقع المحسوبة محليًا.

الكود التالي يُعدِّد المدخلات التي ستُستخدم في الأمثلة اللاحقة. النوع المتوقع للـ observables هو [`qiskit.quantum_info.SparsePauliOp`](../api/qiskit/qiskit.quantum_info.SparsePauliOp). لاحظ أن الدائرة في المثال مُحددة المعاملات، لكن يمكنك أيضًا تشغيل الـ Estimator على دوائر غير مُحددة المعاملات.

> **Note:** أي دائرة تُمرَّر إلى Estimator يجب **ألا** تحتوي على أي **قياسات**.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# circuit for which you want to obtain the expected value
circuit = QuantumCircuit(2)
circuit.ry(Parameter("theta"), 0)
circuit.h(0)
circuit.cx(0, 1)
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/5b41a52d-8f15-4ce4-b3f6-effd91946d9c-0.svg" alt="Output of the previous code cell" />

In [2]:
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# observable(s) whose expected values you want to compute

observable = SparsePauliOp(["II", "XX", "YY", "ZZ"], coeffs=[1, 1, -1, 1])

# value(s) for the circuit parameter(s)
parameter_values = [[0], [np.pi / 6], [np.pi / 2]]

> **Tip:** يتطلب سير عمل Qiskit Runtime primitives تحويل الدوائر والـ observables لتستخدم فقط التعليمات التي يدعمها QPU (يُشار إليها بـ *دوائر وobservables بنية مجموعة التعليمات (ISA)*). لا تزال الـ primitives المرجعية تقبل التعليمات المجردة لأنها تعتمد على محاكاة statevector محلية، لكن transpiling الدائرة قد يظل مفيدًا من حيث تحسين الدائرة.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(circuit)
>   isa_observable = observable.apply_layout(isa_circuit.layout)
>   ```

### تهيئة Estimator
أنشئ نسخة من [`qiskit.primitives.StatevectorEstimator`](../api/qiskit/qiskit.primitives.StatevectorEstimator).

In [3]:
from qiskit.primitives import StatevectorEstimator

estimator = StatevectorEstimator()

### التشغيل والحصول على النتائج
يستخدم هذا المثال دائرة واحدة فقط (من نوع [`QuantumCircuit`](../api/qiskit/qiskit.circuit.QuantumCircuit)) وobservable واحد.

شغِّل التقدير عن طريق استدعاء الطريقة [`StatevectorEstimator.run`](../api/qiskit/qiskit.primitives.StatevectorEstimator#run)، التي تُعيد نسخة من كائن [`PrimitiveJob`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveJob). يمكنك الحصول على النتائج من الوظيفة (كـكائن [`qiskit.primitives.PrimitiveResult`](../api/qiskit/qiskit.primitives.PrimitiveResult))
باستخدام الطريقة [`qiskit.primitives.PrimitiveJob.result`](../api/qiskit/qiskit.primitives.PrimitiveJob#result).

In [4]:
job = estimator.run([(circuit, observable, parameter_values)])
result = job.result()
print(f" > Result class: {type(result)}")

 > Result class: <class 'qiskit.primitives.containers.primitive_result.PrimitiveResult'>


#### Get the expected value from the result

The primitives result outputs an array of [`PubResult`](/docs/api/qiskit/qiskit.primitives.PubResult#pubresult) objects, where each item of the array is a `PubResult` object that contains in its data the array of evaluations corresponding to every circuit-observable combination in the PUB.

To retrieve the expectation values and metadata for the first (and in this case, only) circuit evaluation, we must access the evaluation [`data`](/docs/api/qiskit/qiskit.primitives.PubResult#data) for PUB 0:

In [5]:
print(f" > Expectation value: {result[0].data.evs}")
print(f" > Metadata: {result[0].metadata}")

 > Expectation value: [4.         3.73205081 2.        ]
 > Metadata: {'target_precision': 0.0, 'circuit_metadata': {}}


#### الحصول على قيمة التوقع من النتيجة
تُخرج نتائج الـ primitives مصفوفة من كائنات [`PubResult`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#pubresult)، حيث كل عنصر في المصفوفة هو كائن `PubResult` يحتوي في بياناته على مصفوفة التقييمات المقابلة لكل تركيبة دائرة-observable في الـ PUB.

لاسترداد قيم التوقع والبيانات الوصفية لتقييم الدائرة الأولى (والوحيدة في هذه الحالة)، يجب الوصول إلى [`data`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult#data) للـ PUB 0:

In [6]:
# Estimate expectation values for two PUBs, both with 0.05 precision.
precise_job = estimator.run(
    [(circuit, observable, parameter_values)], precision=0.05
)

For a full example, see the [Estimator examples](/docs/guides/estimator-examples) page.

## Use the reference Sampler

The reference implementations of `SamplerV2` in `qiskit.primitives` is the [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler) class. It takes circuits and parameters as inputs and returns the results from sampling from the output probability distributions as a quasi-probability distribution of output states.

The following code prepares the inputs used in the examples that follow. Note that
these examples run a single parametrized circuit, but you can also run Sampler
on non-parametrized circuits.

In [7]:
from qiskit import QuantumCircuit

circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw("mpl", style="iqp")

<Image src="../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg" alt="Output of the previous code cell" />

### ضبط خيارات تشغيل Estimator
بشكل افتراضي، يُنفِّذ الـ Estimator المرجعي حسابًا دقيقًا للـ statevector استنادًا إلى فئة
[`quantum_info.Statevector`](../api/qiskit/qiskit.quantum_info.Statevector).
لكن يمكن تعديل ذلك لإدخال تأثير الحمل الزائد للأخذ بالعينات (المعروف أيضًا بـ "shot noise").

يقبل الـ Estimator وسيطة `precision` تُعبِّر عن أشرطة الخطأ التي يجب أن يستهدفها تطبيق الـ primitive لتقديرات قيم التوقع. هذا هو الحمل الزائد للأخذ بالعينات ويُعرَّف حصريًا في الطريقة `.run()`. يتيح لك ذلك ضبط الخيار بدقة حتى مستوى الـ PUB.

In [8]:
from qiskit.primitives import StatevectorSampler

sampler = StatevectorSampler()

للاطلاع على مثال كامل، راجع صفحة [أمثلة Primitives](primitives-examples#estimator-examples).

## استخدام Sampler المرجعي
التطبيق المرجعي لـ `SamplerV2` في `qiskit.primitives` هو فئة [`StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler). يأخذ الدوائر والمعاملات كمدخلات ويُعيد نتائج أخذ العينات من توزيعات الاحتمالات للمخرجات كتوزيع شبه احتمالي لحالات المخرجات.

الكود التالي يُعدِّد المدخلات المستخدمة في الأمثلة اللاحقة. لاحظ أن هذه الأمثلة تُشغِّل دائرة واحدة محددة المعاملات، لكن يمكنك أيضًا تشغيل الـ Sampler على دوائر غير محددة المعاملات.

In [9]:
# execute 1 circuit with Sampler
job = sampler.run([circuit])
pub_result = job.result()[0]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-sdk-primitives/extracted-outputs/d4c0ac3b-8e5b-4cde-bb26-256324982c2c-0.svg)

> **Note:** أي دائرة كمية تُمرَّر إلى Sampler **يجب** أن تحتوي على قياسات.

> **Tip:** يتطلب سير عمل Qiskit Runtime primitives تحويل الدوائر لتستخدم فقط التعليمات التي يدعمها QPU (يُشار إليها بدوائر ISA). لا تزال الـ primitives المرجعية تقبل التعليمات المجردة لأنها تعتمد على محاكاة statevector محلية، لكن transpiling الدائرة قد يظل مفيدًا من حيث تحسين الدائرة.
> 
>   ```python
>   # Generate a pass manager without providing a backend
>   from qiskit.transpiler import generate_preset_pass_manager
> 
>   pm = generate_preset_pass_manager(optimization_level=1)
>   isa_circuit = pm.run(qc)
>   ```

### تهيئة `SamplerV2`
أنشئ نسخة من [`qiskit.primitives.StatevectorSampler`](../api/qiskit/qiskit.primitives.StatevectorSampler):

In [10]:
from qiskit.transpiler import generate_preset_pass_manager

# create two circuits
circuit1 = circuit.copy()
circuit2 = circuit.copy()

# transpile circuits
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit1 = pm.run(circuit1)
isa_circuit2 = pm.run(circuit2)
# execute 2 circuits using Sampler
job = sampler.run([(isa_circuit1), (isa_circuit2)])
pub_result_1 = job.result()[0]
pub_result_2 = job.result()[1]
print(f" > Result class: {type(pub_result)}")

 > Result class: <class 'qiskit.primitives.containers.sampler_pub_result.SamplerPubResult'>


### التشغيل والحصول على النتائج

In [11]:
# Define quantum circuit with 2 qubits
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()
circuit.draw()

        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
meas: 2/══════════════╩══╩═
                      0  1 

In [12]:
# Transpile circuit
pm = generate_preset_pass_manager(optimization_level=1)
isa_circuit = pm.run(circuit)
# Run using Sampler
result = sampler.run([circuit]).result()
# Access result data for PUB 0
data_pub = result[0].data
# Access bitstring for the classical register "meas"
bitstrings = data_pub.meas.get_bitstrings()
print(f"The number of bitstrings is: {len(bitstrings)}")
# Get counts for the classical register "meas"
counts = data_pub.meas.get_counts()
print(f"The counts are: {counts}")

The number of bitstrings is: 1024
The counts are: {'11': 495, '00': 529}


تقبل الـ Primitives عدة PUBs كمدخلات، وتحصل كل PUB على نتيجتها الخاصة. لذلك يمكنك تشغيل دوائر مختلفة مع تركيبات متنوعة من المعاملات والـ observables، واسترداد نتائج الـ PUB:

In [13]:
# Sample two circuits at 128 shots each.
sampler.run([isa_circuit1, isa_circuit2], shots=128)
# Sample two circuits at different amounts of shots. The "None"s are necessary
# as placeholders
# for the lack of parameter values in this example.
sampler.run([(isa_circuit1, None, 123), (isa_circuit2, None, 456)])

For a full example, see the [Sampler examples](/docs/guides/sampler-examples) page.
## Next steps

<Admonition type="tip" title="Recommendations">
  - For higher-performance simulation that can handle larger circuits, or to incorporate noise models into your simulation, see [Exact and noisy simulation with Qiskit Aer primitives](simulate-with-qiskit-aer).
  - To learn how to use Quantum Composer for simulation, see the [IBM Quantum Composer](/docs/guides/composer) guide.
  - Read the [Qiskit Estimator API](/docs/api/qiskit/1.4/qiskit.primitives.Estimator) reference.
  - Read the [Qiskit Sampler API](/docs/api/qiskit/1.4/qiskit.primitives.Sampler) reference.
  - Read [Migrate to V2 primitives](/docs/guides/v2-primitives).
</Admonition>